# Lesson LIN 1: Vectors, Neural State Space, & The Dot Product Duality [SOLUTIONS GUIDE]
**Foundations · Applied Linear Algebra for Neural Arrays**  
*Pedagogical Structure: 3Blue1Brown Geometric Duality + Electrophysiological Rigor*

---

> **INSTRUCTOR / SOLUTIONS GUIDE**: This notebook contains the verified reference implementations, adversarially tested math, and expected outputs for Lesson LIN 1.

## 1. Physical Mental Model: Electrodes as Basis Vectors
In electrophysiology, think of each **recording channel** as an axis in a high-dimensional **Neural State Space**:
- An electrode in the **Subthalamic Nucleus (STN)** is basis vector $\hat{e}_{1} = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$.
- An electrode in the **Globus Pallidus internus (GPi)** is basis vector $\hat{e}_{2} = \begin{bmatrix} 0 \\ 1 \end{bmatrix}$.
- At any time sample $t$, the measured voltages form a single coordinate vector in state space:
  $$\vec{v}(t) = v_{\text{STN}}(t) \hat{e}_{1} + v_{\text{GPi}}(t) \hat{e}_{2} = \begin{bmatrix} v_{\text{STN}}(t) \\ v_{\text{GPi}}(t) \end{bmatrix}$$

### Critical Biological Reality: Channels Are NOT Independent Basis Vectors
In pure linear algebra, coordinate axes are orthogonal by construction. But in real neural tissue:
1. **Volume Conduction**: Transmembrane currents spread through the conductive brain volume, so nearby contacts record the same underlying dipolar sources.
2. **The Shared Reference**: Monopolar recordings share a common reference electrode (e.g. cannula or scalp). Any motion or noise on that reference appears on every channel simultaneously.

This is why **Guardrail G1 (Monopolar Common-Mode)** exists in this codebase: it blocks a recipe requesting a `monopolar` montage on a shared-reference recording, so that reference motion is not read as widespread brain synchronization. Its severity is `block, overridable`, because an explicit montage comparison is a legitimate reason to ask for monopolar anyway.

In Lesson LIN 2 we will see what re-referencing actually is: **a linear map applied to the channel vector**, chosen so the shared reference term cancels. Note carefully two things it is not. It is **not a change of basis**, because the CAR matrix is singular with rank $C-1$; it projects onto a subspace and destroys one dimension permanently, which is why Lesson LIN 3 has to reach for the pseudo-inverse. And it does **not diagonalize the covariance matrix**. Removing a shared additive term is not the same as decorrelating channels, and volume conduction leaves substantial off-diagonal covariance behind even after CAR. Diagonalizing a covariance matrix is what eigendecomposition does, and that is Lesson LIN 5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configure publication-grade styling
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.edgecolor'] = '#404040'
plt.rcParams['axes.linewidth'] = 0.8

print('Environment initialized successfully!')

---
### STEP 1: Vector Length (Euclidean LIN 2 Norm)
The length of a vector $\vec{v} = [v_1, v_2, \dots, v_N]$ is given by the Pythagorean theorem generalized to $N$ dimensions:
$$\|\vec{v}\| = \sqrt{\sum_{i=1}^N v_i^2} = \sqrt{v_1^2 + v_2^2 + \dots + v_N^2}$$

**Task 1**: Implement `compute_vector_norm(v)` from scratch using `np.sum` and `np.sqrt` without using `np.linalg.norm`.

In [ ]:
def compute_vector_norm(v: np.ndarray) -> float:
    """Compute the Euclidean (LIN 2) norm of vector v from scratch."""
    return float(np.sqrt(np.sum(v ** 2)))


In [ ]:
# --- TEST CELL FOR STEP 1 ---
test_v1 = np.array([3.0, 4.0])
expected_1 = 5.0  # 3-4-5 right triangle
test_v2 = np.array([1.0, 2.0, 2.0])
expected_2 = 3.0  # sqrt(1 + 4 + 4) = 3

assert np.isclose(compute_vector_norm(test_v1), expected_1), f'Failed on 2D: expected {expected_1}'
assert np.isclose(compute_vector_norm(test_v2), expected_2), f'Failed on 3D: expected {expected_2}'
print('✅ Step 1 Passed! Vector norm is working correctly.')

---
### STEP 2: The Algebraic Dot Product
The algebraic dot product between two vectors $\vec{a}$ and $\vec{b}$ of length $N$ is the sum of their element-wise products:
$$\vec{a} \cdot \vec{b} = \sum_{i=1}^N a_i b_i = \vec{a}^T \vec{b}$$

**Task 2**: Implement `compute_dot_product(a, b)` from scratch using `np.sum` without using `np.dot` or `@`.

In [ ]:
def compute_dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """Compute the dot product between vectors a and b from scratch."""
    return float(np.sum(a * b))


In [ ]:
# --- TEST CELL FOR STEP 2 ---
a_test = np.array([2.0, 3.0, -1.0])
b_test = np.array([4.0, -2.0, 5.0])
# 2*4 + 3*(-2) + (-1)*5 = 8 - 6 - 5 = -3
expected_dot = -3.0

assert np.isclose(compute_dot_product(a_test, b_test), expected_dot), f'Expected {expected_dot}'
print('✅ Step 2 Passed! Algebraic dot product is working correctly.')

---
### STEP 3: The 3Blue1Brown Projection Duality (Honest Non-Circular Verification)
The dot product connects algebra to geometry:
$$\vec{a} \cdot \vec{b} = \|\vec{a}\| \|\vec{b}\| \cos(\theta)$$

**The Non-Circular Proof**:
To test this honestly without circular logic, we must calculate the geometric angle $\theta$ **independently of the dot product** using polar trigonometry:
$$\theta_a = \text{atan2}(a_y, a_x), \quad \theta_b = \text{atan2}(b_y, b_x), \quad \theta_{\text{trig}} = |\theta_b - \theta_a|$$
Then we evaluate $\|\vec{a}\| \|\vec{b}\| \cos(\theta_{\text{trig}})$ and compare against the algebraic sum $\sum a_i b_i$.

**Zero-Norm Handling**: If either vector has zero length, angle $\theta$ is undefined; `compute_cosine_similarity` must return `np.nan`.

**Task 3**: Implement `compute_cosine_similarity(a, b)` using your Step 1 and Step 2 functions.

In [ ]:
def compute_cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute normalized cosine similarity cos(theta) in range [-1.0, +1.0].
    Returns np.nan if either vector has zero norm (angle is undefined).
    """
    norm_a = compute_vector_norm(a)
    norm_b = compute_vector_norm(b)
    if norm_a == 0.0 or norm_b == 0.0:
        return np.nan
    return compute_dot_product(a, b) / (norm_a * norm_b)


In [ ]:
# --- TEST CELL FOR STEP 3: INDEPENDENT NON-CIRCULAR VERIFICATION ---
vec_a = np.array([30.0, 40.0])
vec_b = np.array([50.0, 10.0])

# 1. Compute algebraic dot product
alg_dot = compute_dot_product(vec_a, vec_b)

# 2. Compute angle INDEPENDENTLY via trigonometry (atan2)
theta_a = np.arctan2(vec_a[1], vec_a[0])
theta_b = np.arctan2(vec_b[1], vec_b[0])
theta_trig = np.abs(theta_b - theta_a)

# 3. Compute geometric dot product from the independent angle
geom_dot = compute_vector_norm(vec_a) * compute_vector_norm(vec_b) * np.cos(theta_trig)

# 4. Verify they match with machine precision
assert np.isclose(alg_dot, geom_dot), f'Mismatch: alg={alg_dot}, geom={geom_dot}'

# 5. Pin the VALUE of the cosine similarity, not merely its edge case.
#    cos(theta_trig) comes from atan2 and never touches compute_cosine_similarity,
#    so this checks the result against something independent of it.
cos_measured = compute_cosine_similarity(vec_a, vec_b)
assert np.isclose(cos_measured, np.cos(theta_trig)), \
    f'cosine similarity must equal cos(theta): got {cos_measured}, expected {np.cos(theta_trig)}'

# 6. The three properties that normalization actually buys. An implementation
#    that forgets to divide by the norms passes none of them.
assert np.isclose(compute_cosine_similarity(vec_a, vec_a), 1.0), 'a vector is identical to itself'
assert np.isclose(compute_cosine_similarity(vec_a, -vec_a), -1.0), 'an inverted vector gives -1'
assert np.isclose(cos_measured, compute_cosine_similarity(7.3 * vec_a, vec_b)), \
    'cosine similarity must not change when either vector is rescaled'

# 7. Check zero-norm edge case returns np.nan
zero_v = np.array([0.0, 0.0])
assert np.isnan(compute_cosine_similarity(vec_a, zero_v)), 'Zero norm must return np.nan'

print(f'Algebraic Dot Product (sum a_i * b_i): {alg_dot:.4f}')
print(f'Geometric Dot Product (||a||*||b||*cos θ_trig): {geom_dot:.4f}')
print(f'Independent Angle θ: {np.rad2deg(theta_trig):.2f}°')
print('✅ Step 3 Passed! Duality confirmed non-circularly via independent trigonometry.')

---
## 4. Neural Case Study 1: Subthalamic Beta Burst Detection
In Parkinson's disease, the subthalamic nucleus (STN) exhibits pathological bursts of 13–30 Hz beta oscillations (as configured in `configs/bands.yaml`).

Real neural LFP has a **$1/f$ aperiodic spectral decay (pink noise)**: low frequencies have far more power than high frequencies. Below, we synthesize true $1/f$ pink noise plus a planted 20 Hz beta burst.

We project the raw signal onto a frequency bank of in-phase (cosine) and quadrature (sine) templates to compute spectral power:
$$\text{Power}(f) = (\vec{x} \cdot \cos_f)^2 + (\vec{x} \cdot \sin_f)^2$$

*Note*: In production code, use `scipy.signal.welch` or multitaper estimators. Here we write the projection from first principles.

**Task 4**: Compute the dot products against the cosine and sine templates and calculate total power.

In [ ]:
srate = 1000
time = np.linspace(0, 1.0, srate, endpoint=False)

# Generate true 1/f pink noise via spectral 1/sqrt(f) filtering
np.random.seed(42)
white = np.random.randn(len(time))
freqs = np.fft.rfftfreq(len(time), 1.0 / srate)
freqs[0] = 1.0  # avoid divide-by-zero at DC
pink_fft = np.fft.rfft(white) / np.sqrt(freqs)
pink_noise = np.fft.irfft(pink_fft, n=len(time))
pink_noise = (pink_noise / np.std(pink_noise)) * 2.0

# Plant 20 Hz beta burst (13-30 Hz band per configs/bands.yaml)
beta_burst = 2.5 * np.sin(2 * np.pi * 20 * time)
raw_lfp = beta_burst + pink_noise

test_frequencies = np.arange(5, 61, 1)


def scan_power_profile(signal):
    """Project `signal` onto a quadrature template pair at each test frequency.

    Summing the SQUARED cosine and sine projections is what makes the result
    independent of the phase of the oscillation. One projection alone would report
    near-zero power for a real oscillation that started in the wrong phase.
    Production equivalent: `scipy.signal.welch`.
    """
    profile = []
    for freq in test_frequencies:
        template_cos = np.cos(2 * np.pi * freq * time)
        template_sin = np.sin(2 * np.pi * freq * time)
        dot_cos = compute_dot_product(signal, template_cos)
        dot_sin = compute_dot_product(signal, template_sin)
        power = (dot_cos ** 2) + (dot_sin ** 2)
        profile.append(power)
    return np.array(profile)


power_profile = scan_power_profile(raw_lfp)

print('Spectral scan complete! Run next cell to plot your power spectrum.')


In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(test_frequencies, power_profile, color='#06b6d4', lw=2.5, label='Measured Spectral Power')
peak_freq = test_frequencies[np.argmax(power_profile)]
plt.axvline(20, color='#f59e0b', ls='--', lw=2, label=f'Planted 20 Hz Beta Peak (Detected: {peak_freq} Hz)')
plt.xlabel('Frequency (Hz)', fontweight='bold')
plt.ylabel('Projection power (μV²·samples², unnormalized)', fontweight='bold')
plt.title('Case Study 1: Subthalamic Beta Burst Isolated from 1/f Pink Noise', fontweight='bold')
plt.legend(frameon=True)
plt.grid(True, alpha=0.3, ls='--')
plt.show()

assert peak_freq == 20, f'Expected peak at 20 Hz, got {peak_freq}'

# The assert above only locates the peak, and the peak LOCATION is identical
# whether you squared the projections, took their square root, or used just one
# of them. Three properties pin down the quantity itself.
i20 = int(np.flatnonzero(test_frequencies == 20)[0])

# 1. Power is quadratic in amplitude. An amplitude would give 2.0 here.
ratio = scan_power_profile(2.0 * raw_lfp)[i20] / power_profile[i20]
assert np.isclose(ratio, 4.0, rtol=1e-6), \
    f'doubling the signal must multiply power by 4.0, got {ratio:.3f}. ' \
    'A ratio of 2.0 means you computed amplitude and called it power.'

# 2. Power is independent of the oscillation's phase. This is the entire reason
#    for summing the squared cosine AND sine projections instead of one of them.
power_sin_phase = scan_power_profile(beta_burst)[i20]
power_cos_phase = scan_power_profile(2.5 * np.cos(2 * np.pi * 20 * time))[i20]
assert np.isclose(power_sin_phase, power_cos_phase, rtol=1e-6), \
    f'a quadrature pair makes power phase-invariant: got {power_sin_phase:.4g} for a sine ' \
    f'and {power_cos_phase:.4g} for a cosine. Dropping either projection makes the answer ' \
    'depend on where the oscillation happened to start.'

# 3. Against the closed form for a sinusoid of amplitude A over N samples: (A*N/2)^2.
analytic = (2.5 * len(time) / 2.0) ** 2
assert np.isclose(power_sin_phase, analytic, rtol=1e-6), \
    f'expected {analytic:.6g}, got {power_sin_phase:.6g}'

print(f'Doubling the signal multiplies power by {ratio:.1f}, and phase changes it by '
      f'{abs(power_sin_phase - power_cos_phase):.2e}.')
print('✅ Case Study 1 Confirmed! 20 Hz beta peak successfully isolated.')

---
## 5. Neural Case Study 2: Matched Filtering on High-Density Probes (Neuropixels)
Neuropixels probes record action potentials in the **AP band (0.3 to 10 kHz)** sampled at 30 kHz. A typical extracellular unit waveform lasts $\sim 1.5\text{ ms}$ (45 samples).

### Why Pure Cosine Similarity Fails for Spike Detection
Cosine similarity is scale-invariant: $\cos(c \vec{w}, \vec{w}) = 1.0$. A tiny $0.5\ \mu\text{V}$ noise ripple matching the shape scores $0.9$, triggering a false detection!

**The Matched Filter (Kilosort / Production approach)**: We normalize the template to a unit vector ($\hat{w} = \vec{w} / \|\vec{w}\|$), mean-center each local data window, and compute the **unnormalized projection**:
$$s(t) = \tilde{x}(t) \cdot \hat{w}$$
This preserves amplitude information: large action potentials yield large projections, while low-amplitude noise ripples stay below threshold.

*Note*: In production, compute cross-correlation using `scipy.signal.correlate` or FFT-based convolution.

**Task 5**: Slide a 45-sample window across the 1000-sample stream (total of $1000 - 45 + 1 = 956$ windows) and compute the matched filter projection.

In [ ]:
# Create synthetic 45-sample biphasic action potential template
spike_t = np.linspace(-1, 1, 45)
unit_template = -np.exp(-spike_t**2 / 0.08) + 0.35 * np.exp(-(spike_t - 0.45)**2 / 0.12)
unit_template = unit_template - np.mean(unit_template)
unit_template = unit_template / compute_vector_norm(unit_template)

# Simulate 1000 samples of 30 kHz Neuropixels data with 3 planted spikes
np.random.seed(99)
stream = np.random.normal(0, 0.4, 1000)
planted_spikes = [180, 480, 780]
for idx in planted_spikes:
    stream[idx:idx+45] += unit_template * 35.0

n_windows = len(stream) - len(unit_template) + 1
matched_filter_trace = np.zeros(n_windows)

for i in range(n_windows):
    window = stream[i : i + 45]
    window_centered = window - np.mean(window)
    matched_filter_trace[i] = compute_dot_product(window_centered, unit_template)

print(f'Processed {len(matched_filter_trace)} windows. Run next cell to plot detections!')


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Raw Stream
ax1.plot(stream, color='#737373', lw=1.2, label='Raw Neuropixels AP Band (0.3-10 kHz)')
for p in planted_spikes:
    ax1.axvspan(p, p + 45, color='#fde047', alpha=0.35, label='Planted Unit' if p == 180 else '')
ax1.set_ylabel('Voltage (μV)')
ax1.set_title('Case Study 2: Action Potential Matched Filtering on Neuropixels Probe', fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3, ls='--')

# Matched Filter Trace
ax2.plot(matched_filter_trace, color='#10b981', lw=1.8, label='Matched Filter Projection (Dot Product)')
detection_thresh = 15.0
ax2.axhline(detection_thresh, color='#ef4444', ls='--', lw=1.5, label=f'Threshold ({detection_thresh} μV)')
ax2.set_xlabel('Sample Index (Time)')
ax2.set_ylabel('Projection (μV)')
ax2.set_title('Matched Filter Output: Clear Detection with Zero False Alarms', fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3, ls='--')

plt.tight_layout()
plt.show()

# Count detection EVENTS, not supra-threshold samples. The trace stays above
# threshold for a run of samples around each spike, so a sample count is not a
# spike count, and `len(...) > 0` would pass on a single stray sample anywhere.
above = matched_filter_trace > detection_thresh
onsets = np.flatnonzero(np.diff(above.astype(int)) == 1) + 1
if above[0]:
    onsets = np.r_[0, onsets]

assert len(onsets) == len(planted_spikes), \
    f'expected {len(planted_spikes)} detection events, got {len(onsets)} ' \
    f'({int(above.sum())} individual samples cross the threshold)'

# "Zero false alarms" is in the title above, so test it rather than assert it.
for onset in onsets:
    nearest = min(abs(int(onset) - p) for p in planted_spikes)
    assert nearest <= len(unit_template), f'false alarm at sample {onset}'

print(f'{len(onsets)} events from {int(above.sum())} supra-threshold samples, no false alarms.')

# A note on the mean-centering in the previous cell, which is worth understanding
# rather than copying. It is a no-op HERE: `unit_template` was itself mean-centered,
# so subtracting the window mean subtracts a constant times a vector that sums to
# zero. Verify that rather than take it on faith.
w = stream[planted_spikes[0]:planted_spikes[0] + len(unit_template)]
assert np.isclose(compute_dot_product(w - np.mean(w), unit_template),
                  compute_dot_product(w, unit_template)), 'centering should not matter here'
print('Window centering changes nothing while the template is zero-mean.')
print('It matters the moment the template is not, which is why both are centered.')
print('✅ Case Study 2 Confirmed! Matched filtering isolated all 3 units.')

---
## 6. The Scientific Trap: Shared Reference DC Drift & The Pearson Link

### The Mathematical Trap
Suppose Channel 1 (STN) and Channel 2 (GPi) are recorded against a **shared reference electrode** that has an amplifier DC drift of $+50\ \mu\text{V}$.
Even if the biological activity on both channels is completely independent (orthogonal):
$$x_1(t) = s_1(t) + 50, \quad x_2(t) = s_2(t) + 50$$
The common offset pushes both vectors into the positive orthant: $\vec{x}_1 \cdot \vec{x}_2 \approx N \cdot (50)^2$. Cosine similarity shoots to near $+1.0$, creating a massive false positive!

### The Solution: Mean-Centering (The Pearson Correlation)
When you subtract the mean from each channel:
$$\tilde{x} = x - \bar{x}, \quad \tilde{y} = y - \bar{y}$$
The cosine similarity between the mean-centered vectors equals the exact **Pearson Correlation Coefficient**:
$$r = \frac{(\vec{x} - \bar{x}) \cdot (\vec{y} - \bar{y})}{\|\vec{x} - \bar{x}\| \|\vec{y} - \bar{y}\|}$$

**Task 6**: Implement `compute_pearson_correlation(x, y)` by mean-centering inputs before calling your `compute_cosine_similarity` function.

In [ ]:
def compute_pearson_correlation(x: np.ndarray, y: np.ndarray) -> float:
    """Compute Pearson correlation coefficient by mean-centering and applying cosine similarity."""
    x_centered = x - np.mean(x)
    y_centered = y - np.mean(y)
    return compute_cosine_similarity(x_centered, y_centered)


In [ ]:
# --- TEST CELL FOR STEP 6 ---
np.random.seed(42)
sig_a = np.random.randn(200)
sig_b = np.random.randn(200)

# Both channels contaminated by a +50.0 uV shared reference DC drift
sig_a_drift = sig_a + 50.0
sig_b_drift = sig_b + 50.0

false_sim = compute_cosine_similarity(sig_a_drift, sig_b_drift)
true_r = compute_pearson_correlation(sig_a_drift, sig_b_drift)
numpy_r = np.corrcoef(sig_a_drift, sig_b_drift)[0, 1]

print(f'Raw Cosine Similarity WITH DC Drift : {false_sim:+.4f} (FALSE POSITIVE: Artifactually High!)')
print(f'Your Pearson Correlation (Centered)  : {true_r:+.4f} (Correct: Near Zero)')
print(f'NumPy Reference Correlation          : {numpy_r:+.4f}')

assert false_sim > 0.95, 'False positive must demonstrate the danger of uncentered data'
assert np.isclose(true_r, numpy_r), 'Your mean-centered correlation must match np.corrcoef exactly!'
print('✅ Step 6 Passed! You have mastered the DC offset trap and the Pearson duality.')

---
## Summary of What You Mastered in Lesson LIN 1
1. **Neural State Space**: An electrode array with $N$ contacts defines an $N$-dimensional coordinate system where brain states are vectors.
2. **Physical Channels are Not Orthogonal**: Volume conduction and shared references introduce massive covariance (Guardrail `G1`).
3. **3Blue1Brown Projection Duality**: The algebraic dot product $\vec{a}^T \vec{b}$ is identical to the geometric projection of $\vec{a}$ onto the 1D axis spanned by $\vec{b}$.
4. **Matched Filtering vs Cosine Similarity**: Normalization strips amplitude; for action potential detection, matched filtering preserves amplitude to reject low-voltage noise.
5. **The DC Offset Trap**: Uncentered DC baselines distort vector angles, which is why mean-centering (Pearson correlation) is mandatory for uncorrupted electrophysiology analysis.

Next up: **Lesson LIN 2 — Matrices as Spatial Operators (Directional DBS Montages, ECoG CAR, and Linear Derivations)**!